In [0]:
-- Silver View 1: Cleaned Transactions
CREATE VIEW IF NOT EXISTS nestle_dev_silver.core.v_sales_transactions_cleaned AS
SELECT 
  transaction_id,
  product_id,
  region,
  channel,
  customer_id,
  quantity,
  unit_price,
  amount,
  dbt_valid_from,
  dbt_valid_to,
  dbt_is_current,
  ROUND(amount / NULLIF(quantity, 0), 2) as effective_unit_price
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE
  AND amount > 0
  AND quantity > 0;

SELECT COUNT(*) as cleaned_rows FROM nestle_dev_silver.core.v_sales_transactions_cleaned;

In [0]:
-- Silver View 2: Customer Summary
CREATE VIEW IF NOT EXISTS nestle_dev_silver.core.v_customer_transaction_summary AS
SELECT 
  customer_id,
  COUNT(*) as transaction_count,
  COUNT(DISTINCT product_id) as unique_products,
  COUNT(DISTINCT region) as regions_purchased,
  SUM(amount) as total_spent,
  AVG(amount) as avg_transaction_value
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE
GROUP BY customer_id;

SELECT COUNT(*) as customer_rows FROM nestle_dev_silver.core.v_customer_transaction_summary;

In [0]:
-- Silver View 3: Product Region Performance
CREATE VIEW IF NOT EXISTS nestle_dev_silver.core.v_product_region_performance AS
SELECT 
  product_id,
  region,
  COUNT(*) as transaction_count,
  SUM(quantity) as total_quantity,
  SUM(amount) as total_revenue,
  AVG(amount) as avg_amount
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE
GROUP BY product_id, region;

SELECT COUNT(*) as product_region_rows FROM nestle_dev_silver.core.v_product_region_performance;